# NIICU Multispectral Dataset — Full Preprocessing, Cleaning & Augmentation Pipeline
**Model:** YOLOv11-RGBT

###  Folder Structure
```
dataset/
├── visible/
│   ├── train/
│   └── test/
├── infrared/
│   ├── train/
│   └── test/
└── labels/
    ├── train/
    └── test/
```

## 1. Imports

In [ ]:
import cv2
import os
import yaml
import shutil
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

## 2. Configuration

In [ ]:
ROOT = Path("dataset")
OUT  = Path("dataset_cleaned")

IMG_SIZE = 640
SEED = 42
random.seed(SEED)

splits = ["train", "test"]

CLASS_NAMES = ["person"]

## 3. Create Output Folders

In [ ]:
for split in splits:
    for folder in ["visible", "infrared", "labels"]:
        (OUT / folder / split).mkdir(parents=True, exist_ok=True)

print("Folders Ready")

## 4. Helper Functions

In [ ]:
def valid_image(path):
    img = cv2.imread(str(path))
    return img is not None

def resize_image(img):
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE))

def save_img(path, img):
    cv2.imwrite(str(path), img)

## 5. Label Conversion — TSV (NIICU format) → YOLO Format

Original NIICU TSV columns: `x1  y1  x2  y2  type  occluded  bad`

In [ ]:
def bbox_to_yolo(x1, y1, x2, y2, w, h):
    xc = ((x1 + x2) / 2) / w
    yc = ((y1 + y2) / 2) / h
    bw = (x2 - x1) / w
    bh = (y2 - y1) / h
    return xc, yc, bw, bh


def convert_label_tsv_to_yolo(tsv_file, out_txt, img_w=640, img_h=512):
    df = pd.read_csv(
        tsv_file,
        sep="\t",
        header=None,
        names=["x1", "y1", "x2", "y2", "type", "occluded", "bad"]
    )

    lines = []
    for _, row in df.iterrows():
        xc, yc, bw, bh = bbox_to_yolo(
            row.x1, row.y1, row.x2, row.y2, img_w, img_h
        )
        lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")

    with open(out_txt, "w") as f:
        f.write("\n".join(lines))


# Example usage:
# convert_label_tsv_to_yolo("sample.tsv", "sample.txt")

## 6. Data Augmentation Functions

In [ ]:
def horizontal_flip(img):
    return cv2.flip(img, 1)

def vertical_flip(img):
    return cv2.flip(img, 0)

def adjust_brightness(img, alpha=1.2, beta=15):
    return cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

def gaussian_noise(img):
    noise = np.random.normal(0, 10, img.shape).astype(np.uint8)
    return cv2.add(img, noise)

def blur(img):
    return cv2.GaussianBlur(img, (5, 5), 0)

def clahe_thermal(img):
    gray  = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(2.0, (8, 8))
    gray  = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

## 7. YOLO Label Transform After Flip

In [ ]:
def flip_yolo_label(txt_path, out_path, mode="h"):
    lines = open(txt_path).read().strip().split("\n")
    new_lines = []

    for line in lines:
        vals = line.split()
        cls, x, y, w, h = vals
        x, y, w, h = map(float, [x, y, w, h])

        if mode == "h":
            x = 1 - x
        if mode == "v":
            y = 1 - y

        new_lines.append(f"{cls} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")

    open(out_path, "w").write("\n".join(new_lines))

## 8. Main Cleaning + Preprocessing + Augmentation Loop

In [ ]:
for split in splits:

    vis_dir = ROOT / "visible"  / split
    ir_dir  = ROOT / "infrared" / split
    lb_dir  = ROOT / "labels"   / split

    vis_map = {f.stem: f for f in vis_dir.glob("*.*")}
    ir_map  = {f.stem: f for f in ir_dir.glob("*.*")}
    lb_map  = {f.stem: f for f in lb_dir.glob("*.txt")}

    common = sorted(set(vis_map) & set(ir_map) & set(lb_map))

    print(f"\n{split}: matched pairs = {len(common)}")

    for stem in tqdm(common):

        vis_path = vis_map[stem]
        ir_path  = ir_map[stem]
        lb_path  = lb_map[stem]

        # Validate images
        if not valid_image(vis_path):
            continue
        if not valid_image(ir_path):
            continue

        vis = cv2.imread(str(vis_path))
        ir  = cv2.imread(str(ir_path))

        # Resize
        vis = resize_image(vis)
        ir  = resize_image(ir)

        # Thermal Enhancement (CLAHE)
        ir = clahe_thermal(ir)

        # --- Save Original Cleaned ---
        save_img(OUT / "visible"  / split / f"{stem}.jpg", vis)
        save_img(OUT / "infrared" / split / f"{stem}.jpg", ir)
        shutil.copy(lb_path, OUT / "labels" / split / f"{stem}.txt")

        # --- Augmentation 1: Horizontal Flip ---
        vis_h = horizontal_flip(vis)
        ir_h  = horizontal_flip(ir)

        save_img(OUT / "visible"  / split / f"{stem}_hf.jpg", vis_h)
        save_img(OUT / "infrared" / split / f"{stem}_hf.jpg", ir_h)
        flip_yolo_label(lb_path, OUT / "labels" / split / f"{stem}_hf.txt", "h")

        # --- Augmentation 2: Brightness + Gaussian Noise ---
        vis_aug = gaussian_noise(adjust_brightness(vis))
        ir_aug  = gaussian_noise(ir)

        save_img(OUT / "visible"  / split / f"{stem}_aug.jpg", vis_aug)
        save_img(OUT / "infrared" / split / f"{stem}_aug.jpg", ir_aug)
        shutil.copy(lb_path, OUT / "labels" / split / f"{stem}_aug.txt")

## 9. Generate `data.yaml`

In [ ]:
data = {
    "path":  ".",
    "train": "visible/train",
    "val":   "visible/test",
    "nc":    1,
    "names": ["person"]
}

with open(OUT / "data.yaml", "w") as f:
    yaml.dump(data, f, sort_keys=False)

print("Done. Cleaned dataset ready.")
print("\nFinal output structure:")
print("""
dataset_cleaned/
├── visible/train
├── infrared/train
├── labels/train
├── visible/test
├── infrared/test
├── labels/test
└── data.yaml
""")

## 10. Comparitive Study between P3 and Midfusion 


In [ ]:
#Midfusion Training
import warnings
warnings.filterwarnings('ignore')
from ultralytics import YOLO

if __name__ == '__main__':
    model = YOLO('ultralytics/cfg/models/11-RGBT/yolo11s-RGBT-midfusion.yaml')
    # model.info(True,True)
    model.load('D:/pre-trained-weights RGBT/LLVIP-yolo11s-RGBT-midfusion-e300-16-pretrained.pt') # loading pretrain weights
    model.train(data=R'D:/dataset_new/data.yaml',
                cache=False,
                imgsz=640,
                epochs=10,
                batch=16,
                close_mosaic=10,
                workers=4,
                device='0',
                optimizer='SGD',  # using SGD
                # lr0=0.002,
                # resume='', # last.pt path
                # amp=False, # close amp
                # fraction=0.2,
                # pairs_rgb_ir=['visible','infrared'] , # default: ['visible','infrared'] , others: ['rgb', 'ir'],  ['images', 'images_ir'], ['images', 'image']
                use_simotm="RGBT",
                channels=4,
                project='runs/LLVIP',
                name='yolo11s-RGBT-midfusion-TMI',
                )

In [ ]:
#P3 Training
import warnings
warnings.filterwarnings('ignore')
from ultralytics import YOLO

if __name__ == '__main__':
    model = YOLO('D:/YOLOv11-RGBT/ultralytics/cfg/models/11-RGBT/yolo11s-RGBT-midfusion-P3.yaml')
    # model.info(True,True)
    model.load('D:/YOLOv11-RGBT/LLVIP-yolo11s-RGBT-midfusion-P3-e300-16-pretrained.pt') # loading pretrain weights
    model.train(data=r'D:/dataset_new/data.yaml',
                cache=False,
                imgsz=640,
                epochs=10,
                batch=16,
                close_mosaic=10,
                workers=4,
                device='0',
                optimizer='SGD',  # using SGD
                # lr0=0.002,
                # resume='', # last.pt path
                # amp=False, # close amp
                # fraction=0.2,
                # pairs_rgb_ir=['visible','infrared'] , # default: ['visible','infrared'] , others: ['rgb', 'ir'],  ['images', 'images_ir'], ['images', 'image']
                use_simotm="RGBT",
                channels=4,
                project='runs/P3',
                name='yolo11s-RGBT-midfusion-P3-TMI',
                )

## 9. Analysis on P3 vs Midfusion


In [ ]:
import os
import glob

# === CONFIG ===
label_dir = 'runs/detect_evaluation/midfusion_p3_eval/labels_midfusion'  # folder with .txt label files
output_file = 'all_labels_combined_midfusion.txt'  # where to write

# === RUN ===
with open(output_file, 'w') as outfile:
    for path in sorted(glob.glob(os.path.join(label_dir, '*.txt'))):
        with open(path, 'r') as infile:
            for line in infile:
                clean_line = line.strip()
                if clean_line:  # skip empty lines
                    outfile.write(clean_line + '\n')  # ensures 1 label per line


In [ ]:
import cv2
import numpy as np

# === CONFIG ===
label_file = 'all_labels_combined_GT.txt'  # your combined label file
image_size = (512, 640)  # height, width 
output_path = 'label_density_map_GT.png'

# === INIT ===
heatmap = np.zeros(image_size, dtype=np.uint16)

# === FUNCTION ===
def draw_box_density(map_img, box):
    x1, y1, x2, y2 = map(int, box)
    x1 = max(0, min(x1, image_size[1]-1))
    y1 = max(0, min(y1, image_size[0]-1))
    x2 = max(0, min(x2, image_size[1]-1))
    y2 = max(0, min(y2, image_size[0]-1))
    map_img[y1:y2, x1:x2] += 1

# === PROCESS ===
with open(label_file, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 5:
            continue  # skip invalid lines
        xc, yc, w, h = map(float, parts[1:5])
        x1 = (xc - w / 2) * image_size[1]
        y1 = (yc - h / 2) * image_size[0]
        x2 = (xc + w / 2) * image_size[1]
        y2 = (yc + h / 2) * image_size[0]
        draw_box_density(heatmap, [x1, y1, x2, y2])

# === NORMALIZE & SAVE ===
normalized = np.clip(heatmap / heatmap.max(), 0, 1) * 255
colored = cv2.applyColorMap(normalized.astype(np.uint8), cv2.COLORMAP_JET)
cv2.imwrite(output_path, colored)
print(f"Saved density map to: {output_path}")


## 11. Training Midfusion  


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from ultralytics import YOLO

if __name__ == '__main__':

    model = YOLO(
        'ultralytics/cfg/models/11-RGBT/yolo11s-RGBT-midfusion.yaml'
    )

    model.train(
        data=r'D:/dataset_new/data.yaml',
        cache=False,

        imgsz=640,
        epochs=200,
        patience=50,
        batch=16,

        workers=6,
        device='0',

        optimizer='SGD',
        lr0=0.002,
        lrf=0.01,
        momentum=0.937,
        weight_decay=0.0005,

        close_mosaic=10,

        pretrained=r'D:/YOLOv11-RGBT-master/100epoch_midfusion_b16/weights/last.pt',

        amp=True,
        val=True,
        plots=True,

        save=True,
        save_period=25,

        project='runs/200epoch_midfusion_b16/',
        name='yolo11s-RGBT-midfusion-TMI25',

        use_simotm='RGBT',
        channels=4,
        pairs_rgb_ir=['visible', 'infrared'],

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        translate=0.1,
        scale=0.5,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.0,
        erasing=0.4
    )